In [ ]:
# AWS S3 Vectors Integration Lab (STUDENT)
# Instructions: fill the 6 <fill_in the line here> placeholders, then run cells top-to-bottom.


In [ ]:
from pathlib import Path

# <fill_in the line here>
# (Hint: import boto3 and SentenceTransformer)
<fill_in the line here>


In [ ]:
# <fill_in the line here>
# (Hint: set your AWS access key id as a string)
AWS_ACCESS_KEY_ID = <fill_in the line here>

# <fill_in the line here>
# (Hint: set your AWS secret access key as a string)
AWS_SECRET_ACCESS_KEY = <fill_in the line here>

AWS_REGION = "us-east-1"
BUCKET_NAME = "airline-policy-vectors-YOURNAME"
VECTOR_INDEX_NAME = "airline-policy-index"
TOP_K = 3


In [ ]:
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)
client = session.client("s3vectors")

print("Connected to s3vectors in", AWS_REGION)


In [ ]:
bucket_response = client.list_vector_buckets()
entries = bucket_response.get("vectorBuckets") or []
bucket_names = [x.get("vectorBucketName") for x in entries if x.get("vectorBucketName")]

print("Vector buckets found:", len(bucket_names))
print("Configured bucket visible:", BUCKET_NAME in bucket_names)
bucket_names[:10]


In [ ]:
index_response = client.list_indexes(vectorBucketName=BUCKET_NAME)
index_entries = index_response.get("indexes") or []
index_names = [x.get("indexName") for x in index_entries if x.get("indexName")]

print("Indexes found:", len(index_names))
print("Expected index visible:", VECTOR_INDEX_NAME in index_names)
index_names


In [ ]:
# <fill_in the line here>
# (Hint: use the same model from instructor notebook)
MODEL_NAME = <fill_in the line here>
model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)


In [ ]:
# <fill_in the line here>
# (Hint: point to airline_security_policy.txt in this folder)
policy_path = <fill_in the line here>
policy_text = policy_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in policy_text.split("\n\n") if p.strip()]

print("Paragraphs loaded:", len(paragraphs))
paragraphs[:2]


In [ ]:
embeddings = model.encode(paragraphs)
embedding_vectors = embeddings.tolist()
embedding_dim = len(embedding_vectors[0])

print("Embedding dimension:", embedding_dim)
print("Vectors prepared:", len(embedding_vectors))


In [ ]:
index_info = client.get_index(
    vectorBucketName=BUCKET_NAME,
    indexName=VECTOR_INDEX_NAME,
)
index_cfg = index_info.get("index") or {}
index_dim = index_info.get("dimension") or index_cfg.get("dimension")
index_dim = index_dim or index_info.get("vectorDimension")
index_dim = index_dim or index_cfg.get("vectorDimension")

if not index_dim:
    raise KeyError(f"Missing dimension in get_index response: {list(index_info.keys())}")
index_dim = int(index_dim)
if index_dim != embedding_dim:
    raise ValueError(
        f"Index dimension {index_dim} != embedding dimension {embedding_dim}. "
        f"Delete index '{VECTOR_INDEX_NAME}' and recreate it with dimension {embedding_dim}."
    )
print("Dimension check passed:", index_dim)


In [ ]:
vectors_payload = []
for i, text in enumerate(paragraphs, start=1):
    vectors_payload.append(
        {
            "key": f"section-{i}",
            "data": {"float32": embedding_vectors[i - 1]},
            "metadata": {"text": text},
        }
    )

print("Payload size:", len(vectors_payload))


In [ ]:
put_response = client.put_vectors(
    vectorBucketName=BUCKET_NAME,
    indexName=VECTOR_INDEX_NAME,
    vectors=vectors_payload,
)

print("put_vectors completed")
put_response


In [ ]:
def query_policy(query_text: str, top_k: int = TOP_K):
    # <fill_in the line here>
    # (Hint: encode query_text and convert vector to list)
    query_vector = <fill_in the line here>

    response = client.query_vectors(
        vectorBucketName=BUCKET_NAME,
        indexName=VECTOR_INDEX_NAME,
        topK=top_k,
        queryVector={"float32": query_vector},
        returnMetadata=True,
        returnDistance=True,
    )
    return response.get("vectors") or []

def pretty_print_results(question: str, top_k: int = TOP_K, preview_chars: int = 200):
    print("Question:", question)
    hits = query_policy(question, top_k=top_k)

    for rank, row in enumerate(hits, start=1):
        text = (row.get("metadata") or {}).get("text", "")
        preview = text[:preview_chars].replace("\n", " ")
        print(f"\n--- Match {rank} ---")
        print("Key:", row.get("key"))
        print("Distance:", row.get("distance"))
        print(preview)


In [ ]:
# Q1
pretty_print_results("What security checks happen before boarding?", top_k=3)


In [ ]:
# Q2
pretty_print_results("What items are prohibited in hand baggage?", top_k=3)


In [ ]:
# Q3
pretty_print_results("What is the process for identity verification at the airport?", top_k=3)
